# Joint Prior

In [ ]:
import superstats as sup

Next, we create a `JointPrior` object to specify priors for each observation model parameter. Superstats distinguishes between the following parameter types:

- **`local_param`** — time-varying and estimated (e.g., the trajectory produced by a `StochasticTransition`)
- **`hyper_param`** — time-invariant and estimated (e.g., the hyperpriors governing a transition, such as `sigma` or `slope`)
- **`deterministic_param`** — time-varying but not estimated (produced by a `DeterministicTransition`)
- **`shared_param`** — time-invariant and estimated (specified via a `Prior`, shared across all time steps)
- **`fixed_param`** — time-invariant and not estimated (a fixed `scalar`)

Each parameter type reflects a different combination of two properties: whether the parameter changes over time (**time-varying** vs. **time-invariant**) and whether it is inferred from data (**estimated**) or fixed (**not estimated**). The `JointPrior` object collects these specifications for all parameters into a single, coherent prior over the full model.

```python
joint_prior = sup.JointPrior(
    param_1: sup.transition.RandomWalk(...), # StochasticTransition
    param_2: sup.transition.Linear(...),     # DeterministicTransition
    param_3: sup.Prior(...),                 # Shared parameter
    param_4: 0.5                             # Fixed parameter
)
```

## Stochastic Transitions

Superstats has implemented the following stochastic transition models:

- `RandomWalk`: with hyperparameters `sigma` for the scale of Gaussian noise and `delta` for linear additive drift.
- `AutoRegression`: an AR(1) process with hyperparameters `sigma` for Gaussian noise, `phi` for the autoregressive coefficient, and `delta` for additive drift.
- `OrnsteinUhlenbeck`: a mean-reverting process with hyperparameters `sigma` for diffusion noise, `mu` for the long-run mean, and `theta` for the mean-reversion speed.
- `LevyFlight`: a random walk with alpha-stable noise, with hyperparameters `sigma` for noise scale, `delta` for additive drift, and `alpha` for tail heaviness/stability.
- `Jump`: a jump process with `p_jump` controlling the probability of jumping to a new proposal value at each step.
- `GaussianProcess`: trajectories sampled from a Gaussian process using configurable RBF, linear, periodic, or composite kernels.
- `Mixture`: a mixture of two or more stochastic transition models, with Dirichlet-distributed mixture weights (does not work with `GaussianProcess`).

For each hyperparameter, we can either specify a prior and estimate it from data, or fix it to a scalar value and exclude it from inference. When nothing is specified, Superstats uses sensible defaults, see `superstats.defaults.transition_defaults`.

The following additional arguments can be specified:

- `bounds` : Lower and upper bounds for the parameter. Parameters are generated in the unconstraint space and then constrained to this range via a scaled sigmoid.
- `initial_prior` : Prior or fixed scalar for the initial parameter value. Note that the initial value is in the unconstrained space and then transformed via scaled sigmoid.

**Example.**


```python
prior = sup.JointPrior(
    param_1 = sup.transition.RandomWalk(
        bounds=(-6, 6),
        inital_prior=sup.Prior("normal", loc=0, scale=2),
        sigma=sup.Prior("halfnormal", scale=0.2)
    ),
    param_2 = ...
)
```

Note that parameter names should match the names used as arguments in the obeservation model simulator.

### Mixture

The `Mixture` transition allows mixing the following stochastic transition models:

- `RandomWalk`
- `AutoRegression`
- `OrnsteinUhlenbeck`
- `LevyFlight`
- `Jump`

The most interesting and sensible mixture is between one of the first four transitions and a `Jump`, since this lets a parameter follow smooth, gradual dynamics most of the time while occasionally undergoing a sudden discrete jump.

A few notes on using the `Mixture` transition:

- `bounds` and `initial_prior` must be defined once at initialization of the `Mixture` itself, and must **not** be specified again within the individual transitions it contains.

- When a `Jump` transition is included in the `Mixture`, its `p_jump` is automatically fixed to $1.0$, since the `mixture_weights` already govern the probability of a jump occurring at a given time step. It would not be sensible for the `Jump` component to be selected at a given step and then, due to `p_jump < 1.0`, have a chance of no jump actually occurring.

- As noted earlier, parameter trajectories are generated in an unconstrained space and then transformed via a scaled sigmoid to ensure they remain within `bounds`. This means that setting `proposal_prior=sup.prior.Prior(dist="logistic", loc=0, scale=1)` for the `Jump` component results in a uniform distribution over the bounds after transformation — i.e., when a jump occurs, the parameter can take on any value within its bounds with equal probability.

**Example.**

```python
prior = sup.JointPrior(
    param_1 = sup.transition.RandomWalk(
        bounds=(-6, 6),
        initial_prior=sup.Prior("normal", loc=0, scale=2),
        sigma=sup.Prior("halfnormal", scale=0.2)
    ),
    param_2 = sup.transition.Mixture(
        bounds=(0, 1),
        initial_prior=sup.Prior("normal", loc=0, scale=1),
        transitions=[
            sup.transition.OrnsteinUhlenbeck(
                sigma=sup.Prior("halfnormal", scale=0.1),
                theta=sup.Prior("halfnormal", scale=0.05)
            ),
            sup.transition.Jump(
                proposal_prior=sup.Prior("logistic", loc=0, scale=1)
            ),
        ],
        mixture_weights=sup.Prior("dirichlet", alpha=[9, 1])
    ),
    param_3 = ...
)
```

## Deterministic Transitions

Coming soon...

## Priors

For all hyperparameters, and whenever we want to estimate a time-invariant observation model parameter, we can specify a standard prior via the `Prior` class. Superstats implements the following distributions:

- **`normal`** — Gaussian, parameterized by `loc` (mean) and `scale` (standard deviation).
- **`halfnormal`** — folded Gaussian, non-negative, parameterized by `scale`. Commonly used for scale/noise parameters (e.g., `sigma`).
- **`uniform`** — uniform over `[low, high]`.
- **`beta`** — Beta distribution on `[0, 1]`, parameterized by shape parameters `a` and `b`.
- **`logistic`** — logistic distribution, parameterized by `loc` and `scale`. Often used as a `proposal_prior` in `Jump` transitions, since it maps to a uniform distribution over `bounds` after the scaled sigmoid transform.
- **`dirichlet`** — distribution over the simplex, parameterized by a concentration vector `alpha`. Used, for example, to specify `mixture_weights` in a `Mixture` transition.

Every `Prior` additionally supports an optional linear transform of the drawn samples via `scale_factor` and `shift`, i.e., `scale_factor * samples + shift`.

Feel free to open an issue if a prior you'd like to use isn't implemented yet.